# Particionamiento y Muestreo de Big Data con PySpark

## Threat Hunting basado en Machine Learning para detección de TTPs en Kubernetes

### Dataset: Integrated Cloud Security 3Datasets (ICS3D)

Este notebook implementa el proceso de caracterización, particionamiento y extracción de submuestras sobre el dataset `Containers_Dataset.csv`, utilizando PySpark como framework de procesamiento distribuido.

El objetivo es construir reglas de particionamiento capaces de preservar la representatividad estadística del tráfico benigno y malicioso presente en entornos Kubernetes basados en microservicios.

## Carga del dataset

Se carga el archivo `Containers_Dataset.csv` utilizando inferencia automática de esquema para detectar los tipos de datos de cada variable presente en el dataset.

In [2]:
# 1. Carga del dataset

file_path = r"C:\Users\masalin2\Downloads\Containers_Dataset.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Dataset cargado correctamente")
print(f"Total de registros: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")

Dataset cargado correctamente
Total de registros: 3231475
Total de columnas: 87


In [3]:
# Ver esquema del dataset
df.printSchema()

root
 |-- Flow ID: string (nullable = true)
 |-- Src IP: string (nullable = true)
 |-- Src Port: integer (nullable = true)
 |-- Dst IP: string (nullable = true)
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packet: integer (nullable = true)
 |-- Total Bwd packets: integer (nullable = true)
 |-- Total Length of Fwd Packet: double (nullable = true)
 |-- Total Length of Bwd Packet: double (nullable = true)
 |-- Fwd Packet Length Max: double (nullable = true)
 |-- Fwd Packet Length Min: double (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: double (nullable = true)
 |-- Bwd Packet Length Min: double (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double 

## Exploración inicial del dataset

Se realiza una validación preliminar de las columnas relevantes utilizadas posteriormente para el proceso de particionamiento.

In [4]:
# Verificar columnas importantes
df.select(
    "Label",
    "Protocol",
    "Flow Bytes/s",
    "Total Fwd Packet"
).show(10)

+-----+--------+------------------+----------------+
|Label|Protocol|      Flow Bytes/s|Total Fwd Packet|
+-----+--------+------------------+----------------+
|    0|       0|               0.0|              24|
|    0|       6| 7075.620627241754|             347|
|    0|       0|               0.0|              60|
|    0|       0|               0.0|              60|
|    0|       6| 6734.636892821606|             485|
|    0|       0|               0.0|              57|
|    0|       6| 616751.2690355331|               5|
|    0|       6|               0.0|               5|
|    0|       6|3.7344453456644655|              10|
|    0|       6| 13.94062198230764|              12|
+-----+--------+------------------+----------------+
only showing top 10 rows


## Creación de variable categórica: traffic_type

La variable `traffic_type` permite separar el tráfico benigno del tráfico malicioso utilizando la variable `Label` como criterio principal de clasificación.

In [5]:
from pyspark.sql.functions import when, col

# Crear variable traffic_type
df = df.withColumn(
    "traffic_type",
    when(col("Label") == 0, "Benign")
    .otherwise("Malicious")
)

# Verificar resultados
df.select("Label", "traffic_type").show(10)

+-----+------------+
|Label|traffic_type|
+-----+------------+
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
|    0|      Benign|
+-----+------------+
only showing top 10 rows


## Creación de variable categórica: protocol_type

La variable `protocol_type` clasifica el tráfico de acuerdo con el protocolo de transporte utilizado, agrupando los registros en tráfico TCP, UDP y otros protocolos.

In [7]:
# Crear variable protocol_type
df = df.withColumn(
    "protocol_type",
    when(col("Protocol") == 6, "TCP")
    .when(col("Protocol") == 17, "UDP")
    .otherwise("Other")
)

# Verificar resultados
df.select(
    "Protocol",
    "protocol_type"
).show(10)

+--------+-------------+
|Protocol|protocol_type|
+--------+-------------+
|       0|        Other|
|       6|          TCP|
|       0|        Other|
|       0|        Other|
|       6|          TCP|
|       0|        Other|
|       6|          TCP|
|       6|          TCP|
|       6|          TCP|
|       6|          TCP|
+--------+-------------+
only showing top 10 rows


## Distribución inicial de particiones

Se calcula la distribución inicial de registros utilizando las variables `traffic_type` y `protocol_type`, con el objetivo de identificar el comportamiento estadístico general del dataset.

In [8]:
# Ver distribución inicial
df.groupBy(
    "traffic_type",
    "protocol_type"
).count().show()

+------------+-------------+-------+
|traffic_type|protocol_type|  count|
+------------+-------------+-------+
|      Benign|        Other|   5235|
|      Benign|          TCP|2905606|
|      Benign|          UDP|  42450|
|   Malicious|        Other|   2524|
|   Malicious|          UDP|  21221|
|   Malicious|          TCP| 254439|
+------------+-------------+-------+



## Cálculo de probabilidades de ocurrencia

En esta sección se calculan las probabilidades de ocurrencia de cada partición observada dentro del dataset, utilizando el total de registros como referencia estadística.

In [9]:
from pyspark.sql.functions import round

# Total de registros
total_records = df.count()

# Calcular probabilidades
partition_distribution = df.groupBy(
    "traffic_type",
    "protocol_type"
).count()

partition_distribution = partition_distribution.withColumn(
    "probability",
    round(col("count") / total_records, 6)
)

partition_distribution.show()

+------------+-------------+-------+-----------+
|traffic_type|protocol_type|  count|probability|
+------------+-------------+-------+-----------+
|      Benign|        Other|   5235|    0.00162|
|      Benign|          TCP|2905606|   0.899158|
|      Benign|          UDP|  42450|   0.013136|
|   Malicious|        Other|   2524|    7.81E-4|
|   Malicious|          UDP|  21221|   0.006567|
|   Malicious|          TCP| 254439|   0.078738|
+------------+-------------+-------+-----------+



## Cálculo de percentiles para Flow Bytes/s

Debido a la alta variabilidad presente en la variable `Flow Bytes/s`, se utilizan percentiles para discretizar la intensidad del tráfico de red en categorías homogéneas.

In [10]:
# Calcular percentiles reales para Flow Bytes/s

percentiles = df.approxQuantile(
    "Flow Bytes/s",
    [0.33, 0.66],
    0.01
)

print(percentiles)

[15841.905766121878, 66084.97723823975]


## Creación de variable categórica: traffic_intensity

La variable `traffic_intensity` clasifica el tráfico en niveles de baja, media y alta intensidad utilizando percentiles calculados directamente sobre el dataset.

In [11]:
# Crear variable traffic_intensity

df = df.withColumn(
    "traffic_intensity",
    when(col("Flow Bytes/s") <= 15841.905766121878, "Low")
    .when(
        (col("Flow Bytes/s") > 15841.905766121878) &
        (col("Flow Bytes/s") <= 66084.97723823975),
        "Medium"
    )
    .otherwise("High")
)

# Verificar resultados
df.select(
    "Flow Bytes/s",
    "traffic_intensity"
).show(10)

+------------------+-----------------+
|      Flow Bytes/s|traffic_intensity|
+------------------+-----------------+
|               0.0|              Low|
| 7075.620627241754|              Low|
|               0.0|              Low|
|               0.0|              Low|
| 6734.636892821606|              Low|
|               0.0|              Low|
| 616751.2690355331|             High|
|               0.0|              Low|
|3.7344453456644655|              Low|
| 13.94062198230764|              Low|
+------------------+-----------------+
only showing top 10 rows


## Generación de particiones completas

Se construyen las particiones finales utilizando las variables de caracterización definidas previamente. Las particiones representan subconjuntos homogéneos del tráfico de red observados en el dataset.

In [12]:
# Distribución completa de particiones

full_partition_distribution = df.groupBy(
    "traffic_type",
    "protocol_type",
    "traffic_intensity"
).count()

full_partition_distribution = full_partition_distribution.withColumn(
    "probability",
    round(col("count") / total_records, 6)
)

full_partition_distribution.orderBy(
    col("count").desc()
).show(50, truncate=False)

+------------+-------------+-----------------+-------+-----------+
|traffic_type|protocol_type|traffic_intensity|count  |probability|
+------------+-------------+-----------------+-------+-----------+
|Benign      |TCP          |Medium           |1045988|0.323687   |
|Benign      |TCP          |Low              |996581 |0.308398   |
|Benign      |TCP          |High             |863037 |0.267072   |
|Malicious   |TCP          |High             |229279 |0.070952   |
|Benign      |UDP          |High             |36637  |0.011338   |
|Malicious   |TCP          |Low              |20855  |0.006454   |
|Malicious   |UDP          |Low              |11651  |0.003605   |
|Malicious   |UDP          |High             |9405   |0.00291    |
|Benign      |Other        |Low              |5228   |0.001618   |
|Benign      |UDP          |Low              |4705   |0.001456   |
|Malicious   |TCP          |Medium           |4305   |0.001332   |
|Malicious   |Other        |Low              |2524   |7.81E-4 

## Creación de vista temporal SQL

Se genera una vista temporal que permitirá realizar consultas SQL sobre el dataset particionado.

In [19]:
# Crear vista temporal para consultas SQL
df.createOrReplaceTempView("ics3d_flows")

## Función de extracción de submuestras

Se implementa una función capaz de recuperar submuestras de prueba basadas en reglas específicas de particionamiento.

In [14]:
# Función para extraer una submuestra de prueba por regla de particionamiento

def get_partition_sample(traffic_type, protocol_type, traffic_intensity, sample_size=10):
    query = f"""
    SELECT *
    FROM ics3d_flows
    WHERE traffic_type = '{traffic_type}'
      AND protocol_type = '{protocol_type}'
      AND traffic_intensity = '{traffic_intensity}'
    LIMIT {sample_size}
    """
    return spark.sql(query)

## Ejemplos de extracción de submuestras

En esta sección se presentan ejemplos de extracción de registros pertenecientes a distintas reglas de particionamiento, con el objetivo de validar el correcto funcionamiento del proceso de segmentación del dataset.

In [15]:
# Ejemplo 1: Tráfico benigno TCP de baja intensidad
sample_1 = get_partition_sample("Benign", "TCP", "Low", 10)
sample_1.select("Label", "Protocol", "Flow Bytes/s", "traffic_type", "protocol_type", "traffic_intensity").show()

+-----+--------+------------------+------------+-------------+-----------------+
|Label|Protocol|      Flow Bytes/s|traffic_type|protocol_type|traffic_intensity|
+-----+--------+------------------+------------+-------------+-----------------+
|    0|       6| 7075.620627241754|      Benign|          TCP|              Low|
|    0|       6| 6734.636892821606|      Benign|          TCP|              Low|
|    0|       6|               0.0|      Benign|          TCP|              Low|
|    0|       6|3.7344453456644655|      Benign|          TCP|              Low|
|    0|       6| 13.94062198230764|      Benign|          TCP|              Low|
|    0|       6|  7073.79043598635|      Benign|          TCP|              Low|
|    0|       6| 6702.628967617506|      Benign|          TCP|              Low|
|    0|       6|               0.0|      Benign|          TCP|              Low|
|    0|       6|2.2940083459340266|      Benign|          TCP|              Low|
|    0|       6| 8.936118740

In [16]:
# Ejemplo 2: Tráfico malicioso TCP de alta intensidad
sample_2 = get_partition_sample("Malicious", "TCP", "High", 10)
sample_2.select("Label", "Protocol", "Flow Bytes/s", "traffic_type", "protocol_type", "traffic_intensity").show()

+-----+--------+--------------------+------------+-------------+-----------------+
|Label|Protocol|        Flow Bytes/s|traffic_type|protocol_type|traffic_intensity|
+-----+--------+--------------------+------------+-------------+-----------------+
|    4|       6|1.2180758017492712E7|   Malicious|          TCP|             High|
|    4|       6|   2632225.547330288|   Malicious|          TCP|             High|
|    4|       6|   784565.9163987137|   Malicious|          TCP|             High|
|    4|       6|1.5547687861271676E7|   Malicious|          TCP|             High|
|    4|       6|             2.089E7|   Malicious|          TCP|             High|
|    4|       6|   1722517.238646271|   Malicious|          TCP|             High|
|    4|       6|   880434.7826086957|   Malicious|          TCP|             High|
|    4|       6|   859154.9295774647|   Malicious|          TCP|             High|
|    4|       6|1.9777573529411767E7|   Malicious|          TCP|             High|
|   

In [17]:
# Ejemplo 3: Tráfico benigno UDP de alta intensidad
sample_3 = get_partition_sample("Benign", "UDP", "High", 10)
sample_3.select("Label", "Protocol", "Flow Bytes/s", "traffic_type", "protocol_type", "traffic_intensity").show()

+-----+--------+------------------+------------+-------------+-----------------+
|Label|Protocol|      Flow Bytes/s|traffic_type|protocol_type|traffic_intensity|
+-----+--------+------------------+------------+-------------+-----------------+
|    0|      17| 221126.7605633803|      Benign|          UDP|             High|
|    0|      17|190765.49210206562|      Benign|          UDP|             High|
|    0|      17|241319.44444444444|      Benign|          UDP|             High|
|    0|      17|151912.56830601091|      Benign|          UDP|             High|
|    0|      17| 654178.6743515851|      Benign|          UDP|             High|
|    0|      17|203631.64721141374|      Benign|          UDP|             High|
|    0|      17| 666666.6666666667|      Benign|          UDP|             High|
|    0|      17|252411.57556270095|      Benign|          UDP|             High|
|    0|      17|167021.27659574468|      Benign|          UDP|             High|
|    0|      17| 192520.7756